In [1]:
import pytesseract
pytesseract.pytesseract.tesseract_cmd = "/opt/homebrew/bin/tesseract"

In [ ]:
import fitz  #
from PIL import Image
import pytesseract
from pathlib import Path
\
pdf_folder = Path("pdfs") 
output_texts = {}  


for pdf_file in pdf_folder.glob("*.pdf"):
    print(f"Processing {pdf_file.name}...")
    doc = fitz.open(pdf_file)
    pdf_text = []

    for page_num, page in enumerate(doc, start=1):
        pix = page.get_pixmap()
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

        # OCR
        text = pytesseract.image_to_string(img)
        lines = [line.strip() for line in text.split("\n") if line.strip()]
        pdf_text.extend(lines)

    output_texts[pdf_file.stem] = pdf_text

    
for pdf_name, lines in output_texts.items():
    print(f"\n--- {pdf_name} ---")
    print("\n".join(lines[:10]))  # checking the first 10 lines


Processing week_49.pdf...
Processing week_48.pdf...
Processing week_4.pdf...
Processing week_29.pdf...
Processing week_15.pdf...
Processing week_14.pdf...
Processing week_28.pdf...
Processing week_5.pdf...


In [ ]:
import pdfplumber
import pandas as pd
from pathlib import Path

pdf_folder = Path("pdfs")  
all_rows = []

for pdf_file in pdf_folder.glob("week_*.pdf"):
    week_name = pdf_file.stem
    print(f"Processing {week_name}...")

    with pdfplumber.open(pdf_file) as pdf:
        if len(pdf.pages) < 4:
            print(f"{week_name} has less than 4 pages, skipping.")
            continue

        page = pdf.pages[3]  # look at page 4 specifically (table 1 is on page 4)
        text = page.extract_text()

    lines = text.split("\n")

    # Locating Table 1
    try:
        start_idx = next(i for i, line in enumerate(lines) if "Table 1" in line)
    except StopIteration:
        print(f"Table 1 not found in {week_name}")
        continue

    # Table 1: header + last row
    table_lines = lines[start_idx+1:]
    table_rows = [line.strip() for line in table_lines if line.strip() != ""]

    if len(table_rows) < 2:
        print(f"Not enough rows in Table 1 for {week_name}")
        continue

    import re
    header = re.split(r'\s{2,}', table_rows[0])
    last_row = re.split(r'\s{2,}', table_rows[-1])

    # Add week(2023XX format) column
    header.insert(0, "week")
    last_row.insert(0, week_name)

    all_rows.append((header, last_row))


if all_rows:
    df = pd.DataFrame([row[1] for row in all_rows], columns=all_rows[0][0])
    csv_path = Path("table1_first_last.csv").resolve()
    df.to_csv(csv_path, index=False)
    print(f"Saved CSV to {csv_path}")
else:
    print("No Table 1 data found in any PDF.")
